In [6]:
import time
import pandas as pd
import requests


In [ ]:
import pandas as pd
from pathlib import Path

championship_audit = pd.read_csv(
    "../data/raw/bigwest_2021_2026_audit.csv",
    )


Existing audit rows: 5303


In [4]:
api_url = (
    "https://api.parse.bot/scraper/"
    "6e80f113-decc-4a5f-960a-70b2f1d469bc/"
    "get_meet_team_scores"
)
headers = {
    "X-API-Key": "pmx_e2173f90c664df059780119f36d518e4"
}
new_datasets = {
    2019: 62142,
}

In [7]:
rows = []
successful_years = []

for year, meet_id in new_datasets.items():

    try:
        response = requests.get(
            api_url,
            headers=headers,
            params={"meet_id": str(meet_id)},
            timeout=120
        )

        response.raise_for_status()
        result = response.json()

        if result.get("status") != "success":
            print(f"Failed to scrape {year}")
            continue

        data = result["data"]

        if not all(
            "events" in data.get(gender, {})
            for gender in ["men", "women"]
        ):
            print(f"Incomplete data for {year}; skipping")
            continue

        year_rows = []

        for gender in ["men", "women"]:
            for event_data in data[gender]["events"]:

                event_name = event_data["event"]
                event_lower = event_name.lower()

                if "relay" in event_lower:
                    event_type = "relay"
                    mark_unit = "seconds"

                elif any(x in event_lower for x in [
                    "heptathlon", "decathlon"
                ]):
                    event_type = "combined"
                    mark_unit = "points"

                elif any(x in event_lower for x in [
                    "jump", "vault", "shot put",
                    "discus", "hammer", "javelin"
                ]):
                    event_type = "field"
                    mark_unit = "meters"

                else:
                    event_type = "track"
                    mark_unit = "seconds"

                for entry in event_data["results"]:

                    athletes = entry.get("athletes") or []
                    
                    athlete = (
                        athletes[0]
                        if athletes and event_type != "relay"
                        else {}
                    )

                    year_rows.append({
                        "athlete_id": athlete.get("athlete_id"),
                        "athlete_name": athlete.get("name"),
                        "team": entry.get("team"),
                        "meet_name": data.get("meet_name"),
                        "meet_year": year,
                        "gender": gender,
                        "event": event_name,
                        "mark": entry.get("mark"),
                        "mark_unit": mark_unit,
                        "place": entry.get("place"),
                        "wind": entry.get("wind"),
                        "event_type": event_type,
                        "points": entry.get("points")
                    })

        if not year_rows:
            print(f"No event results for {year}; skipping")
            continue

        rows.extend(year_rows)
        successful_years.append(year)

        print(f"Scraped {year}: {len(year_rows)} rows")

    except (requests.RequestException, ValueError) as e:
        print(f"Error scraping {year}: {e}")

    time.sleep(3.25)


Scraped 2019: 799 rows


In [15]:
new_results = pd.DataFrame(rows)
new_results["athlete_id"] = (
    new_results["athlete_id"].astype("string")
)

championship_audit = pd.concat(
    [championship_audit, new_results],
    ignore_index=True
)

championship_audit = (
    championship_audit
    .drop_duplicates()
    .sort_values(
        ["meet_year", "gender", "event"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)
championship_audit.to_csv(
    "../data/raw/bigwest_2021_2026_audit.csv",
    index=False
)

Cleaning Big west